<a href="https://colab.research.google.com/github/Scarfaced007/NLP_Assignmnets-/blob/main/Assign_2_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Assignment Rule: Must include installation steps for all required dependencies
!pip install nltk bert-score pandas torch transformers scikit-learn

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import time
import math
import nltk
from bert_score import score
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

# Ensure NLTK resources are available for BLEU scoring
nltk.download('punkt')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Deep Learning Environment Initialized. Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.9 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Deep Learning Environment Initialized. Using device: cuda


In [ ]:
print("Downloading XLM-RoBERTa Tokenizer...")
# XLM-R natively understands Devanagari subwords, saving our model from learning character spelling
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
vocab_size = tokenizer.vocab_size
PAD_IDX = tokenizer.pad_token_id
BOS_IDX = tokenizer.bos_token_id
EOS_IDX = tokenizer.eos_token_id

print(f"Tokenizer loaded. Shared Vocabulary Size: {vocab_size:,}")

# Strictly align datasets using Source_id
def load_translation_data(sa_path, en_path):
    df_sa = pd.read_csv(sa_path)
    df_en = pd.read_csv(en_path)
    aligned_df = pd.merge(df_sa, df_en, on='Source_id', how='inner').sort_values('Source_id').reset_index(drop=True)
    return aligned_df

class TranslationDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=64):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        sa_text = str(self.df.iloc[idx]['Sentence_sa'])
        en_text = str(self.df.iloc[idx]['Sentence_en'])

        src_encoded = self.tokenizer(sa_text, max_length=self.max_len, padding='max_length', truncation=True, return_tensors="pt")
        tgt_encoded = self.tokenizer(en_text, max_length=self.max_len, padding='max_length', truncation=True, return_tensors="pt")

        src_ids = src_encoded['input_ids'].squeeze()
        tgt_ids = tgt_encoded['input_ids'].squeeze()

        # Seq2Seq shifted inputs
        decoder_input = tgt_ids[:-1]
        expected_output = tgt_ids[1:]

        return src_ids, decoder_input, expected_output

print("Preparing DataLoaders...")
train_df = load_translation_data('train_sa_10000.csv', 'train_en_10000.csv')
dev_df = load_translation_data('dev_sa_1000.csv', 'dev_en_1000.csv')
test_df = load_translation_data('test_sa_1000.csv', 'test_en_1000.csv')

train_loader = DataLoader(TranslationDataset(train_df, tokenizer), batch_size=32, shuffle=True)
dev_loader = DataLoader(TranslationDataset(dev_df, tokenizer), batch_size=32, shuffle=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded. Shared Vocabulary Size: 250,002
Preparing DataLoaders...


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class CustomSanskritTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=3, dim_feedforward=512, dropout=0.1):
        super(CustomSanskritTransformer, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        # Custom Architecture optimized for extreme parameter efficiency
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead, num_encoder_layers=num_layers,
            num_decoder_layers=num_layers, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.fc_out = nn.Linear(d_model, vocab_size)

    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, src, tgt, tgt_mask=None, src_padding_mask=None, tgt_padding_mask=None):
        src_emb = self.pos_encoder(self.embedding(src))
        tgt_emb = self.pos_encoder(self.embedding(tgt))

        outs = self.transformer(
            src_emb, tgt_emb, tgt_mask=tgt_mask,
            src_key_padding_mask=src_padding_mask, tgt_key_padding_mask=tgt_padding_mask
        )
        return self.fc_out(outs)

In [ ]:
print("Initializing the Custom Micro-Transformer...")
model = CustomSanskritTransformer(vocab_size=vocab_size).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {total_params:,} (Optimized for Efficiency Score)")

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)

EPOCHS = 15
PATIENCE = 3
best_val_loss = float('inf')
epochs_no_improve = 0

print("\nStarting NMT Training...")
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0

    for src, tgt_in, tgt_expected in train_loader:
        src, tgt_in, tgt_expected = src.to(device), tgt_in.to(device), tgt_expected.to(device)

        tgt_mask = model.generate_square_subsequent_mask(tgt_in.size(1)).to(device)
        src_padding_mask = (src == PAD_IDX).to(device)
        tgt_padding_mask = (tgt_in == PAD_IDX).to(device)

        optimizer.zero_grad()
        output = model(src, tgt_in, tgt_mask=tgt_mask, src_padding_mask=src_padding_mask, tgt_padding_mask=tgt_padding_mask)

        output = output.contiguous().view(-1, output.shape[-1])
        tgt_expected = tgt_expected.contiguous().view(-1)

        loss = criterion(output, tgt_expected)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()

    avg_train_loss = epoch_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for src, tgt_in, tgt_expected in dev_loader:
            src, tgt_in, tgt_expected = src.to(device), tgt_in.to(device), tgt_expected.to(device)
            tgt_mask = model.generate_square_subsequent_mask(tgt_in.size(1)).to(device)
            output = model(src, tgt_in, tgt_mask=tgt_mask, src_padding_mask=(src == PAD_IDX).to(device), tgt_padding_mask=(tgt_in == PAD_IDX).to(device))
            loss = criterion(output.contiguous().view(-1, output.shape[-1]), tgt_expected.contiguous().view(-1))
            val_loss += loss.item()

    avg_val_loss = val_loss / len(dev_loader)
    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_nmt_model.pth')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch+1}!")
            break

model.load_state_dict(torch.load('best_nmt_model.pth'))

Initializing the Custom Micro-Transformer...
Total Trainable Parameters: 132,205,714 (Optimized for Efficiency Score)

Starting NMT Training...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch [01/15] - Train Loss: 6.5630 | Val Loss: 5.6499
Epoch [02/15] - Train Loss: 5.1755 | Val Loss: 5.1374
Epoch [03/15] - Train Loss: 4.6121 | Val Loss: 4.8588
Epoch [04/15] - Train Loss: 4.1825 | Val Loss: 4.6826
Epoch [05/15] - Train Loss: 3.8179 | Val Loss: 4.5721
Epoch [06/15] - Train Loss: 3.4884 | Val Loss: 4.5146
Epoch [07/15] - Train Loss: 3.1911 | Val Loss: 4.5968
Epoch [08/15] - Train Loss: 2.9088 | Val Loss: 4.6496
Epoch [09/15] - Train Loss: 2.6454 | Val Loss: 4.6329
Early stopping triggered at epoch 9!


<All keys matched successfully>

In [ ]:
def translate_sentence(model, sentence, tokenizer, max_len=64):
    """Autoregressive Greedy Search Decoding"""
    model.eval()
    src_encoded = tokenizer(sentence, return_tensors="pt", max_length=max_len, truncation=True)
    src = src_encoded['input_ids'].to(device)
    src_padding_mask = (src == PAD_IDX).to(device)

    tgt_ids = [BOS_IDX]

    for i in range(max_len):
        tgt_tensor = torch.LongTensor(tgt_ids).unsqueeze(0).to(device)
        tgt_mask = model.generate_square_subsequent_mask(tgt_tensor.size(1)).to(device)

        with torch.no_grad():
            output = model(src, tgt_tensor, tgt_mask=tgt_mask, src_padding_mask=src_padding_mask)

        next_token = output[0, -1, :].argmax().item()
        tgt_ids.append(next_token)

        if next_token == EOS_IDX:
            break

    return tokenizer.decode(tgt_ids, skip_special_tokens=True)

print("Starting inference on the Test Set...")
start_time = time.time()

predicted_translations = []
gold_translations = test_df['Sentence_en'].tolist()

for idx, row in test_df.iterrows():
    sa_sentence = row['Sentence_sa']
    pred_en = translate_sentence(model, sa_sentence, tokenizer)
    predicted_translations.append(pred_en)

    if idx % 500 == 0 and idx > 0:
        print(f"Translated {idx} sentences...")

end_time = time.time()
inference_time = end_time - start_time

print("\n--- OFFICIAL EVALUATION METRICS ---")
print(f"Total Parameters: {total_params:,}")
print(f"Inference Time: {inference_time:.2f} seconds")

# NLTK BLEU (Default)
nltk.download('punkt_tab') # Added to address the LookupError
bleu_scores = [nltk.translate.bleu_score.sentence_bleu([nltk.word_tokenize(g)], nltk.word_tokenize(p))
               for g, p in zip(gold_translations, predicted_translations)]
print(f"Average NLTK BLEU Score: {sum(bleu_scores) / len(bleu_scores):.4f}")

# BERTScore (F1, Rescaled)
print("Calculating BERTScore (this may take a moment)...")
P, R, F1 = score(predicted_translations, gold_translations, lang="en", verbose=False, rescale_with_baseline=True)
print(f"BERTScore (F1 Rescaled): {F1.mean().item():.4f}")

Starting inference on the Test Set...
Translated 500 sentences...

--- OFFICIAL EVALUATION METRICS ---
Total Parameters: 132,205,714
Inference Time: 313.36 seconds


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram o

Average NLTK BLEU Score: 0.0065
Calculating BERTScore (this may take a moment)...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1 Rescaled): 0.0324


In [ ]:
print("Generating final submission.csv file...")

# Format explicitly required: Source_id and Sentence_en
submission_df = pd.DataFrame({
    'Source_id': test_df['Source_id'],
    'Sentence_en': predicted_translations
})

# Save to UTF-8 as mandated by the rules
submission_df.to_csv('submission.csv', index=False, encoding='utf-8')

print("Success! 'submission.csv' has been generated and is ready for the ZIP file.")

Generating final submission.csv file...
Success! 'submission.csv' has been generated and is ready for the ZIP file.
